# YOLOv11 Fine-Tuning Experiment — 06_DEPTHSENSE_STATIC_DATASET

## What this notebook is (and isn't)

This notebook runs a **fine-tuning experiment**: it takes the current production YOLOv11 champion
(`pothole_detector_yolo11s_v22`) and continues training it on `dataset/06_DEPTHSENSE_STATIC_DATASET/`
— the depth-annotated pothole dataset converted to YOLO format earlier in this project (see
[`dataset/06_DEPTHSENSE_STATIC_DATASET/README.txt`](../dataset/06_DEPTHSENSE_STATIC_DATASET/README.txt)
for provenance).

It is deliberately **not** framed as "fine-tuning to make the model better." It is framed as:

> An experiment to determine whether transfer learning from the current production checkpoint
> improves performance on the DepthSense target dataset, while keeping a reproducible record of
> the result.

A fine-tuned model that performs *worse* than the champion is still a useful, valid outcome — it
tells us transfer learning didn't help for this dataset/hyperparameter combination, and that gets
recorded exactly like a success would. See Section 10 for a real example already in this repo of a
fine-tune run that underperformed its baseline.

## How this differs from the other training assets in this folder

| File | Purpose |
|---|---|
| `train_unified.py` | Stage 1: train a fresh model from COCO weights. |
| `fine_tune.py` | Stage 2 (scripted): fine-tune the champion on `dataset/clean_dataset/`. |
| `train.ipynb` | Exploratory notebook covering both stages on `clean_dataset`. |
| **`finetune_depthsense.ipynb`** (this notebook) | Fine-tunes the champion on the depth-annotated `06_DEPTHSENSE_STATIC_DATASET`, and is the first place in this repo that persistently tracks run history and champion status across experiments (see Section 9). |

## Notebook workflow

1. Environment setup
2. Dataset preparation (train/val/test split of the flat `06_DEPTHSENSE_STATIC_DATASET`)
3. Fine-tuning vs. from-scratch hyperparameter rationale
4. Load the starting checkpoint (current YOLOv11 champion)
5. Hyperparameter configuration
6. Run fine-tuning
7. Evaluate on the validation split
8. Model registry — record this run, compare against the champion, optional human-approved promotion
9. Written analysis: fine-tuning vs. training from scratch, different scenarios
10. Summary & next steps

**This notebook is built ready-to-run but has not been executed.** Training on ~1000 images takes
real GPU time; run it yourself inside the `venv-gpu` environment when ready (see Section 10 for
exact steps).

## 1. Environment Setup

Verify the Python environment (this notebook expects to run inside `venv-gpu`, same as
`train.ipynb`), resolve the project root, and import PyTorch / Ultralytics.

In [1]:
import sys
import os
import json
import time
from pathlib import Path
from datetime import datetime, timezone

# ══════════════════════════════════════════════════════════════════════════
# VERIFY ENVIRONMENT
# ══════════════════════════════════════════════════════════════════════════
python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
print(f"Using Python: {python_version} from {sys.executable}")
if "venv-gpu" not in sys.executable.replace("\\", "/"):
    print("⚠️  This does not look like the venv-gpu interpreter. Training will still work on")
    print("   CPU-only environments but will be extremely slow; switch kernels if this is unexpected.")

# Resolve the project root by walking up from cwd looking for the repo's marker
# folders (backend/ + model_training/), rather than assuming cwd == model_training/.
# Jupyter kernels don't always start with cwd set to the notebook's own directory
# (e.g. VS Code can start them at the workspace root instead), so Path.cwd().parent
# is not reliable here.
def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend").is_dir() and (candidate / "model_training").is_dir():
            return candidate
    raise RuntimeError(
        f"Could not locate project root (expected a 'backend' and 'model_training' "
        f"folder somewhere above {start})."
    )


project_root = _find_project_root(Path.cwd())
print(f"Project root: {project_root}")

# backend/ has its own top-level utils.py (LIVEDET's runtime depth/severity helpers),
# which is a different, unrelated module from this notebook's model_training/utils/
# package. If both directories are on sys.path at once, "import utils" resolves
# ambiguously (and in practice grabs backend/utils.py, breaking
# `from utils.dataset_handler import ...` below with "'utils' is not a package").
# So: import backend's config with backend/ on sys.path, then drop it again before
# putting model_training/ on sys.path for the DatasetHandler import in the next cell.
backend_path = str(project_root / "backend")
sys.path.insert(0, backend_path)
try:
    from config import config
    print("✓ Backend config loaded successfully.")
except ImportError:
    config = None
    print("⚠️ Could not load backend config directly (fine — only used for the BEST_MODEL_PATH fallback below).")
finally:
    sys.path.remove(backend_path)

sys.path.insert(0, str(project_root / "model_training"))

Using Python: 3.10.11 from c:\Users\ihsan\Documents\GitHub\ML2\venv-gpu\Scripts\python.exe
Project root: c:\Users\ihsan\Documents\GitHub\ML2
✓ Backend config loaded successfully.


In [2]:
try:
    import torch
    from ultralytics import YOLO
    print("✓ PyTorch version:", torch.__version__)
    print("✓ CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("✓ GPU Device Name:", torch.cuda.get_device_name(0))
    print("✓ Ultralytics YOLO imported successfully.")
except ImportError as e:
    print("Installing dependencies...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics"])
    import torch
    from ultralytics import YOLO
    print("✓ Setup complete!")



✓ PyTorch version: 2.7.1+cu118
✓ CUDA available: True
✓ GPU Device Name: NVIDIA GeForce GTX 1650
✓ Ultralytics YOLO imported successfully.


## 2. Dataset Preparation

`dataset/06_DEPTHSENSE_STATIC_DATASET/` is **flat** — just `images/`, `labels/`, `depth_ground_truth/`,
no train/val/test split (see its README). We reuse the existing
[`DatasetHandler`](utils/dataset_handler.py) class rather than writing new split logic — it already
implements a stratified-by-index `train_test_split` and YOLO folder/`data.yaml` creation.

The split is written to a **separate** output directory
(`dataset/06_DEPTHSENSE_STATIC_DATASET_split/`) so the original flat dataset stays untouched and
reusable for other purposes.

**Reproducibility:** `random_state=42` is passed explicitly (matching `DatasetHandler`'s own
default) so re-running this notebook doesn't reshuffle which images land in validation — required
for metrics to stay comparable across runs of this notebook. The split is also only generated
**once**: if the split directory already exists, it's reused as-is rather than regenerated.

**Leakage check:** a random image-level split is only valid if the 1000 source images are
independent captures, not frames sampled from the same video/burst (which would let near-duplicate
frames land in both train and val, inflating val metrics). Per
`dataset/06_DEPTHSENSE_STATIC_DATASET/README.txt`, the source images are distinct timestamped
captures; only 2 timestamp pairs are Roboflow duplicate-exports of the same capture, and that was
already handled during the dataset conversion. **This check does not automatically carry over** if
this notebook is ever pointed at a different, sequence/video-derived dataset — re-verify independence
first.

In [3]:
from utils.dataset_handler import DatasetHandler


dataset_root = project_root / "dataset" / "06_DEPTHSENSE_STATIC_DATASET"
split_root = project_root / "dataset" / "06_DEPTHSENSE_STATIC_DATASET_split"

handler = DatasetHandler(dataset_root=dataset_root, output_root=split_root)

SPLIT_SEED = 42  # fixed so val composition is stable across notebook re-runs

if (split_root / "images" / "train").exists():
    print(f"✓ Split already exists at {split_root} — reusing it (delete the folder to force a rebuild).")
else:
    print(f"Creating YOLO train/val/test split at {split_root} (seed={SPLIT_SEED}) ...")
    handler.create_yolo_structure(train_ratio=0.8, val_ratio=0.1, test_ratio=0.1)
    handler.split_dataset(
        images_dir=dataset_root / "images",
        labels_dir=dataset_root / "labels",
        train_ratio=0.8,
        val_ratio=0.1,
        test_ratio=0.1,
        random_state=SPLIT_SEED,
    )
    print("✓ Split created.")

data_yaml_path = handler.create_data_yaml(num_classes=1, class_names=["pothole"])
print(f"✓ data.yaml written to: {data_yaml_path}")

report = handler.validate_labels()
print("\nLabel validation report:")
for k, v in report.items():
    if isinstance(v, list):
        print(f"  {k}: {len(v)} entries")
    else:
        print(f"  {k}: {v}")

train_count = len(list((split_root / "images" / "train").glob("*")))
val_count = len(list((split_root / "images" / "val").glob("*")))
test_count = len(list((split_root / "images" / "test").glob("*")))
print(f"\nSplit sizes -> train: {train_count}, val: {val_count}, test: {test_count}")

✓ Split already exists at c:\Users\ihsan\Documents\GitHub\ML2\dataset\06_DEPTHSENSE_STATIC_DATASET_split — reusing it (delete the folder to force a rebuild).
✓ data.yaml written to: c:\Users\ihsan\Documents\GitHub\ML2\dataset\06_DEPTHSENSE_STATIC_DATASET_split\data.yaml

Label validation report:
  total_images: 1000
  total_labels: 1000
  valid_labels: 1097
  invalid_labels: 0
  missing_labels: 0 entries
  invalid_format: 0 entries

Split sizes -> train: 800, val: 100, test: 100


## 3. Fine-Tuning vs. Training-From-Scratch: Hyperparameter Rationale

Fine-tuning starts from a checkpoint that already knows what a pothole looks like
(`pothole_detector_yolo11s_v22`, trained from COCO weights and then tuned on `clean_dataset`).
That changes which hyperparameters make sense compared to `train_unified.py`'s from-scratch
defaults:

| Hyperparameter | From scratch (`train_unified.py`) | Fine-tune (this notebook) | Why it differs |
|---|---|---|---|
| `lr0` | Higher (e.g. 0.01) | Lower (e.g. 0.001) | The backbone's learned features are already useful; a high LR would overwrite them with noisy gradients from a much smaller dataset ("catastrophic forgetting"). |
| `warmup_epochs` | Longer | Shorter (e.g. 3) | Warmup exists to stabilize randomly-initialized weights early on; a pretrained checkpoint doesn't need as much ramp-up. |
| `cos_lr` | Optional | `True` | A smooth cosine decay avoids abrupt LR changes that could kick a converged model out of a good minimum. |
| `freeze` | `0`/unset (train everything) | `>0` (e.g. 10 backbone layers) | Freezing early layers protects general-purpose low-level features (edges, textures) that transfer regardless of dataset, while letting later layers adapt to the new data's specifics. Too high a freeze value under-adapts; too low risks forgetting. |
| `epochs` | More (100+) | Fewer, with tighter `patience` | A pretrained model converges faster on a related task; excessive epochs on a small dataset risks overfitting rather than learning anything new. |

These are exactly the choices `fine_tune.py` and `train.ipynb`'s fine-tuning section already make
for `clean_dataset` — this notebook applies the same reasoning to `06_DEPTHSENSE_STATIC_DATASET`.

## 4. Load the Starting Checkpoint

The starting point is the current **production YOLOv11 champion**, read from `.env`'s
`BEST_MODEL_PATH` (today `runs/base_models/pothole_detector_yolo11s_v22/weights/best.pt`).

If that path is missing, this falls back to the earlier `pothole_detector_yolo11s` base checkpoint
— but that is **not** the same experiment (it wouldn't be "fine-tune the champion further"), so the
fallback prints a loud warning rather than silently substituting a different checkpoint.

In [4]:
CHAMPION_NAME = "pothole_detector_yolo11s_v22"  # matches model_registry.json champions.yolov11
default_checkpoint = project_root / "runs" / "base_models" / CHAMPION_NAME / "weights" / "best.pt"
fallback_checkpoint = project_root / "runs" / "base_models" / "pothole_detector_yolo11s" / "weights" / "best.pt"

env_model_path = None
if config is not None and getattr(config, "BEST_MODEL_PATH", None):
    env_model_path = (project_root / config.BEST_MODEL_PATH).resolve()

candidate = env_model_path if (env_model_path and env_model_path.exists()) else default_checkpoint

if candidate.exists():
    checkpoint_path = candidate
    is_champion_checkpoint = True
    print(f"✓ Loading production champion checkpoint: {checkpoint_path}")
else:
    checkpoint_path = fallback_checkpoint
    is_champion_checkpoint = False
    print("⚠ BEST_MODEL_PATH was not found.")
    print("⚠ Falling back to base YOLO11s checkpoint.")
    print("⚠ This is NOT the current production champion — results below are not")
    print("⚠ a fine-tune-vs-champion comparison.")
    print(f"⚠ Fallback path: {checkpoint_path}")

assert checkpoint_path.exists(), f"No usable checkpoint found at {checkpoint_path}"
print(f"\nStarting checkpoint: {checkpoint_path}")
print(f"Is production champion: {is_champion_checkpoint}")

✓ Loading production champion checkpoint: C:\Users\ihsan\Documents\GitHub\ML2\models\finetuned\pothole_detector_yolo11s_v22\weights\best.pt

Starting checkpoint: C:\Users\ihsan\Documents\GitHub\ML2\models\finetuned\pothole_detector_yolo11s_v22\weights\best.pt
Is production champion: True


## 5. Fine-Tuning Hyperparameter Configuration

Mirrors `fine_tune.py`'s tunable arguments. Adjust these before running Section 6.

In [5]:
ft_output_name = "pothole_detector_yolo11s_depthsense_v1"

ft_epochs = 100          # fine-tune runs typically converge faster than scratch training
ft_batch_size = 8
ft_imgsz = 800            # matches the champion's own training resolution (see args.yaml)
ft_patience = 20
ft_lr0 = 0.001            # low LR to avoid catastrophic forgetting of the champion's features
ft_lrf = 0.01             # final LR = lr0 * lrf
ft_freeze = 10            # freeze the first 10 backbone layers, adapt the rest
ft_device = 0 if torch.cuda.is_available() else "cpu"

# Native Ultralytics augmentations — same values as fine_tune.py, appropriate for a
# depth-camera dataset with different lighting/framing than clean_dataset
ft_mosaic = 1.0
ft_mixup = 0.15
ft_hsv_h = 0.015
ft_hsv_s = 0.7
ft_hsv_v = 0.4

print("Fine-Tuning Configuration:")
print(f"  Output run name: {ft_output_name}")
print(f"  Dataset:         {data_yaml_path}")
print(f"  Checkpoint:      {checkpoint_path}")
print(f"  Epochs:          {ft_epochs}")
print(f"  Batch size:      {ft_batch_size}")
print(f"  Image size:      {ft_imgsz}")
print(f"  Patience:        {ft_patience}")
print(f"  lr0 / lrf:       {ft_lr0} / {ft_lrf}")
print(f"  Freeze layers:   {ft_freeze}")
print(f"  Device:          {ft_device}")

Fine-Tuning Configuration:
  Output run name: pothole_detector_yolo11s_depthsense_v1
  Dataset:         c:\Users\ihsan\Documents\GitHub\ML2\dataset\06_DEPTHSENSE_STATIC_DATASET_split\data.yaml
  Checkpoint:      C:\Users\ihsan\Documents\GitHub\ML2\models\finetuned\pothole_detector_yolo11s_v22\weights\best.pt
  Epochs:          100
  Batch size:      8
  Image size:      800
  Patience:        20
  lr0 / lrf:       0.001 / 0.01
  Freeze layers:   10
  Device:          0


## 6. Run Fine-Tuning

**Not executed in this notebook as delivered** — a full run on ~800 training images takes real GPU
time. Run this cell yourself when ready (see Section 10 for exact steps).

In [6]:
ft_model = YOLO(str(checkpoint_path))
print("Model loaded. Starting fine-tuning...")

ft_start = time.time()
ft_results = ft_model.train(
    data=str(data_yaml_path),
    epochs=ft_epochs,
    imgsz=ft_imgsz,
    batch=ft_batch_size,
    device=ft_device,
    patience=ft_patience,
    save=True,
    project=str(project_root / "runs" / "base_models"),
    name=ft_output_name,
    workers=2,
    close_mosaic=5,
    plots=True,
    freeze=ft_freeze,
    lr0=ft_lr0,
    lrf=ft_lrf,
    warmup_epochs=3,
    cos_lr=True,
    mosaic=ft_mosaic,
    mixup=ft_mixup,
    hsv_h=ft_hsv_h,
    hsv_s=ft_hsv_s,
    hsv_v=ft_hsv_v,
)

ft_duration = time.time() - ft_start
print(f"\nFine-tuning complete in {ft_duration:.0f}s ({ft_duration/3600:.2f} hours)")
print(f"Results saved to: {project_root / 'runs' / 'base_models' / ft_output_name}")

Model loaded. Starting fine-tuning...
New https://pypi.org/project/ultralytics/8.4.120 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.10.11 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=c:\Users\ihsan\Documents\GitHub\ML2\dataset\06_DEPTHSENSE_STATIC_DATASET_split\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, m

## 7. Evaluate on the Validation Split

Beyond mAP50, we also capture recall, inference speed, and file size — the registry's promotion
policy (Section 8) weighs all four, not just mAP50, so a model that's marginally more accurate but
much slower doesn't get silently promoted.

In [7]:
ft_val = ft_model.val(data=str(data_yaml_path))

ft_weights_path = project_root / "runs" / "base_models" / ft_output_name / "weights" / "best.pt"

candidate_metrics = {
    "precision": float(ft_val.results_dict.get("metrics/precision(B)", 0)),
    "recall": float(ft_val.results_dict.get("metrics/recall(B)", 0)),
    "map50": float(ft_val.results_dict.get("metrics/mAP50(B)", 0)),
    "map50_95": float(ft_val.results_dict.get("metrics/mAP50-95(B)", 0)),
    "speed_inference_ms": float(ft_val.speed.get("inference", 0)),
    "file_size_mb": ft_weights_path.stat().st_size / (1024 * 1024) if ft_weights_path.exists() else None,
}

print("DepthSense Fine-Tune — Validation Metrics:")
for k, v in candidate_metrics.items():
    print(f"  {k:<20}: {v}")

Ultralytics 8.4.14  Python-3.10.11 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 259.758.5 MB/s, size: 74.8 KB)
val: Scanning C:\Users\ihsan\Documents\GitHub\ML2\dataset\06_DEPTHSENSE_STATIC_DATASET_split\labels\val.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 1.8s/it 12.4s0.7s
                   all        100        100      0.829       0.92      0.932      0.616
Speed: 7.9ms preprocess, 20.7ms inference, 0.0ms loss, 3.7ms postprocess per image
Results saved to C:\Users\ihsan\Documents\GitHub\ML2\model_training\runs\detect\val
DepthSense Fine-Tune — Validation Metrics:
  precision           : 0.8286763806477859
  recall              : 0.92
  map50               : 0.932488790962634
  map50_95      

## 8. Model Registry — Recording the Run and Comparing to the Champion

[`model_registry.json`](model_registry.json) is a small persistent record of every training/fine-tune
run and which run is the current champion, **tracked separately per architecture** (YOLOv8 and
YOLOv11 are independent lineages — this notebook only ever produces `architecture: "yolov11"`
entries). It's already seeded with the two current champions:

- `yolov8` → `yolov8s_fresh`
- `yolov11` → `pothole_detector_yolo11s_v22`

**Every run gets recorded, regardless of outcome** — a fine-tune that underperforms the champion is
still valid experimental data and stays in `runs` with `status: "experiment"`.

**`models/finetuned/` is never read or written here.** That folder is the user's own manual backup
of champion weights, kept on their own schedule; this notebook only ever points at weights inside
`runs/base_models/`.

In [8]:
REGISTRY_PATH = project_root / "model_training" / "model_registry.json"


def load_registry():
    with open(REGISTRY_PATH, "r") as f:
        return json.load(f)


def save_registry(registry):
    with open(REGISTRY_PATH, "w") as f:
        json.dump(registry, f, indent=2)


def get_champion(registry, architecture):
    champion_name = registry["champions"].get(architecture)
    if champion_name is None:
        return None
    return next((r for r in registry["runs"] if r["name"] == champion_name), None)


def record_run(registry, name, architecture, dataset, base_checkpoint, metrics, weights_path, status="experiment"):
    entry = {
        "name": name,
        "architecture": architecture,
        "status": status,
        "dataset": dataset,
        "base_checkpoint": base_checkpoint,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "metrics": metrics,
        "weights": str(weights_path),
    }
    registry["runs"] = [r for r in registry["runs"] if r["name"] != name]  # replace if re-run
    registry["runs"].append(entry)
    return entry


registry = load_registry()
print(f"Registry loaded: {len(registry['runs'])} run(s) on record.")
print(f"Current champions: {registry['champions']}")

Registry loaded: 2 run(s) on record.
Current champions: {'yolov8': 'yolov8s_fresh', 'yolov11': 'pothole_detector_yolo11s_v22'}


In [9]:
this_run = record_run(
    registry,
    name=ft_output_name,
    architecture="yolov11",
    dataset="06_DEPTHSENSE_STATIC_DATASET",
    base_checkpoint=CHAMPION_NAME if is_champion_checkpoint else "pothole_detector_yolo11s",
    metrics=candidate_metrics,
    weights_path=ft_weights_path.relative_to(project_root),
    status="experiment",
)
save_registry(registry)
print(f"✓ Recorded run '{ft_output_name}' in model_registry.json (status: experiment).")

✓ Recorded run 'pothole_detector_yolo11s_depthsense_v1' in model_registry.json (status: experiment).


In [10]:
def compare_to_champion(candidate_metrics, champion_metrics, policy):
    primary = policy["primary_metric"]
    secondary = policy["secondary_metric"]
    max_latency_regression = policy["max_inference_ms_regression"]

    checks = {}
    checks[f"primary ({primary}) improves"] = candidate_metrics[primary] > champion_metrics[primary]
    checks[f"secondary ({secondary}) does not regress"] = candidate_metrics[secondary] >= champion_metrics[secondary]

    champ_latency = champion_metrics.get("speed_inference_ms")
    cand_latency = candidate_metrics.get("speed_inference_ms")
    if champ_latency is None or cand_latency is None:
        checks["inference latency within budget"] = None  # can't evaluate — champion has no recorded speed yet
    else:
        checks["inference latency within budget"] = (cand_latency - champ_latency) <= max_latency_regression

    verdict = all(v for v in checks.values() if v is not None)
    return verdict, checks


champion = get_champion(registry, "yolov11")
print(f"Champion on record: {champion['name']}")
print(f"Champion metrics:   {champion['metrics']}")
print(f"Candidate metrics:  {candidate_metrics}\n")

verdict, checks = compare_to_champion(candidate_metrics, champion["metrics"], registry["promotion_policy"])

print("Promotion policy checks:")
for name, passed in checks.items():
    symbol = "?" if passed is None else ("PASS" if passed else "FAIL")
    print(f"  [{symbol}] {name}")

print(f"\nOverall recommendation: {'PROMOTE' if verdict else 'DO NOT PROMOTE'}")
print("This is a recommendation only — review it yourself before running the promotion cell below.")

Champion on record: pothole_detector_yolo11s_v22
Champion metrics:   {'precision': 0.6942, 'recall': 0.65505, 'map50': 0.71684, 'map50_95': 0.40865, 'speed_inference_ms': None, 'file_size_mb': 18.32}
Candidate metrics:  {'precision': 0.8286763806477859, 'recall': 0.92, 'map50': 0.932488790962634, 'map50_95': 0.6160533992775202, 'speed_inference_ms': 20.650444000493735, 'file_size_mb': 37.43997669219971}

Promotion policy checks:
  [PASS] primary (map50) improves
  [PASS] secondary (recall) does not regress
  [?] inference latency within budget

Overall recommendation: PROMOTE
This is a recommendation only — review it yourself before running the promotion cell below.


## 8b. Promotion (Human-Approved, Off by Default)

If — after reviewing the verdict above yourself, not just trusting the printout — you agree the new
run should become the YOLOv11 champion, uncomment and run the cell below.

**What this cell does:** updates `model_registry.json` only — marks the new run as `"champion"`,
demotes the previous YOLOv11 champion to `"retired"`, and updates `champions.yolov11`.

**What this cell deliberately does NOT do:**
- it does not copy any weight files into `models/finetuned/` — that stays your own manual backup
  step, on your own schedule;
- it does not edit `.env` — it only prints the `BEST_MODEL_PATH` line for you to apply by hand,
  at the same time you'd update your own backup.

In [11]:
# --- PROMOTION DEACTIVATED BY DEFAULT — REQUIRES MANUAL REVIEW FIRST ---
# To promote this run to YOLOv11 champion, uncomment and run:
#
# if not verdict:
#     print("Verdict was DO NOT PROMOTE — re-check before overriding this.")
# else:
#     for r in registry["runs"]:
#         if r["name"] == champion["name"]:
#             r["status"] = "retired"
#     this_run["status"] = "champion"
#     registry["champions"]["yolov11"] = ft_output_name
#     save_registry(registry)
#     print(f"✓ model_registry.json updated: yolov11 champion is now '{ft_output_name}'.")
#     print("\nTo actually deploy this model, update .env yourself:")
#     print(f"  BEST_MODEL_PATH=./{this_run['weights']}")
#     print("\nRemember: models/finetuned/ backups are managed by you, not this notebook.")

## 9. Fine-Tuning vs. Training-From-Scratch — Written Analysis

This section is intentionally **markdown only, not re-executed code** — it's a scenario-based
discussion for the thesis, grounded in this project's own recorded run history rather than generic
claims.

### What this project's own history actually shows

| Run | Type | mAP@50 | mAP@50-95 | Precision | Recall | Notes |
|---|---|---|---|---|---|---|
| `YOLO11s_Baseline` | From scratch (COCO → `clean_dataset`) | 57.8% | 30.5% | 64.2% | 55.3% | `model_comparison_results/exact_metrics.json` |
| `YOLO11s_FineTuned` (early/exploratory run) | Fine-tune | 33.8% | 18.7% | 56.8% | 33.1% | Same file — an earlier, distinct fine-tune experiment, not the current champion |
| `pothole_detector_yolo11s_v22` (current champion) | Fine-tune | 71.7% | 40.9% | 69.4% | 65.5% | `runs/base_models/pothole_detector_yolo11s_v22/results.csv`, final logged epoch |

The important thing about this table is **not** "fine-tuning is worse" or "fine-tuning is better" —
it contains one fine-tune run that clearly underperformed its baseline, and another fine-tune run
that clearly outperformed it. The two fine-tuned runs differ in dataset split, hyperparameters,
augmentation, training duration, and possibly the starting checkpoint itself, so they are not a
controlled A/B comparison.

**The correct conclusion:** *in the recorded `YOLO11s_FineTuned` experiment, fine-tuning
underperformed the from-scratch baseline. This does not prove fine-tuning is inherently worse than
training from scratch — it shows fine-tuning is not guaranteed to help, and its effect must be
empirically validated per target domain and hyperparameter setting.* That is exactly what this
notebook's Section 8 registry exists to do: no fine-tune result is assumed good until it's measured
and compared against the current champion on its own merits.

### Scenario-by-scenario trade-offs

**Small target dataset (this notebook's case — 06_DEPTHSENSE_STATIC_DATASET, ~1000 images)**
- *Favors fine-tuning.* Training a detector from scratch on ~1000 images risks the backbone never
  learning generalizable low-level features (edges, gradients, textures) before overfitting to the
  small sample. Starting from a checkpoint that already has those features lets the model spend its
  limited data budget adapting to what's actually novel about this dataset (depth-camera framing,
  lighting).
- *Risk:* if the frozen layers encode features that don't transfer well to the new domain, the model
  can underperform a scratch-trained one that at least fits the new data directly — this is plausibly
  part of what happened in the `YOLO11s_FineTuned` row above.

**Domain shift (static fixed-height camera source vs. LIVEDET's moving dashboard camera)**
- `dataset/04_TEST_FROM_DEPTH_MEASURED_SET/README.txt` already flags this: "PothRGBD images were
  captured with a static fixed-height camera. LIVEDET targets a moving dashboard camera." Both
  `06_DEPTHSENSE_STATIC_DATASET` (also PothRGBD-derived) and any from-scratch model trained on it
  inherit this domain gap — neither approach solves it. Fine-tuning doesn't make the gap worse, but
  it doesn't close it either; it should be treated as a known limitation of any model trained (from
  scratch or fine-tuned) on this dataset alone.

**Compute / time budget**
- *Favors fine-tuning.* A pretrained checkpoint typically converges in far fewer epochs than
  training from scratch (the champion's own log stopped at epoch 45/150 via early-stopping
  `patience`). For a thesis project with finite GPU time, fine-tuning gets useful signal faster.

**Catastrophic forgetting**
- *Risk specific to fine-tuning.* If `lr0` is too high or `freeze` too low, gradients from the new,
  smaller dataset can overwrite the general features the checkpoint already learned — net effect:
  worse performance than either training from scratch (which never had those features to lose) or a
  more conservative fine-tune. This is the most direct hypothesis for why `YOLO11s_FineTuned` scored
  below `YOLO11s_Baseline`, and is exactly why Section 3's hyperparameter choices (low LR, `freeze=10`,
  cosine decay) exist.

**When training from scratch is actually the better call**
- A materially different class taxonomy (e.g. `02_DETAILED_CRACKS_ANNOTATION`'s 4 classes vs. this
  project's usual single `pothole` class) where the pretrained detection head doesn't transfer
  cleanly.
- A architecture change (e.g. moving from YOLOv8 to YOLOv11 entirely) where there is no compatible
  checkpoint to fine-tune from.
- A large, diverse dataset is available (thousands of varied images) where overfitting risk is low
  and the benefit of a warm-started backbone becomes marginal relative to the cost of Backbone bias
  toward the pretrained domain.

### Takeaway for this notebook specifically

Because `06_DEPTHSENSE_STATIC_DATASET` is small (~1000 images, favoring fine-tuning) but also
domain-shifted relative to `clean_dataset` (where the champion was tuned) and structurally different
from LIVEDET's real deployment conditions, the outcome is genuinely uncertain in advance — which is
exactly why this is framed as an experiment with a registry, not a foregone conclusion.

## 10. Summary & Next Steps

**To actually run this experiment:**
1. Activate `venv-gpu` and open this notebook with that kernel (same as `train.ipynb`).
2. Run Sections 1–5 top to bottom.
3. Run Section 6 (`ft_model.train(...)`) — this is the long-running step (real GPU time).
4. Run Section 7 to evaluate, and Section 8 to record the run and see the promotion recommendation.
5. **Before uncommenting the Section 8b promotion cell:** actually look at the verdict table, not
   just whether it says PROMOTE — check whether the metric deltas are meaningful or noise-level, and
   whether `speed_inference_ms`/`file_size_mb` regressions matter for your deployment target.
6. If you promote, remember to (a) manually update your own `models/finetuned/` backup, and
   (b) manually update `.env`'s `BEST_MODEL_PATH` — neither happens automatically.

See [`FOLDER_STRUCTURE.txt`](FOLDER_STRUCTURE.txt) for the broader `runs/base_models/` storage
convention this notebook follows, and [`model_registry.json`](model_registry.json) for the full run
history this notebook maintains.